# Jeff Tetris on the ANE (overnight 2026-10-08, round 02)

**Apple-silicon Mac + Core AI required.** This notebook serves the shipped model. For its training history, see [How the shipped Tetris adapter was trained](../docs/TETRIS_TRAINING.md). For a new experiment, see the [Colab training tutorial](03_tetris_training_colab.ipynb) (not yet GPU-validated).

Serve the base build with the Tetris build as an adapter and play greedy games with the browser's spawn/top-out rules, comparing against the
El-Tetris teacher. The full 32-seed benchmark is `python -m projects.tetris_training.play --split test`; this notebook
plays a few seeds (about 45 s per 150-piece game on an M5 Max).

Uses `$JEFF_TETRIS_BUILD` if set (e.g. `~/Models/jeff-tetris-training/overnight-20261008/round-02/build`), else
`$JEFF_MODELS_DIR/tetris`, downloading it if needed.

In [ ]:
import os, sys, time, statistics
from pathlib import Path
sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "jeff_nb.py").exists() else Path.cwd() / "notebooks"))
import jeff_nb as J
print("repo:", J.REPO)
print("models dir:", J.models_dir())
print("COREAI_PYTHON:", os.environ.get("COREAI_PYTHON", "<not set>"))

In [ ]:
GAMES = int(os.environ.get("JEFF_NB_GAMES", "2"))
MAX_PIECES = int(os.environ.get("JEFF_NB_MAX_PIECES", "150"))
base, tetris = J.ensure("base"), J.ensure("tetris")
print("tetris build:", tetris)
if not J.reuse_url():
    for b in (base, tetris):
        J.compile_build(b)
proc, url = J.start_server(base, {"tetris": tetris})   # the server's --model is the base build
print(J.health(url)["backend"], J.health(url)["adapters"])

In [ ]:
from projects.tetris_training.dataset import rollout
from projects.tetris_training.play import ServedPolicy
from jeff_tetris import board_text

adapter = "tetris"
policy = ServedPolicy(url, adapter)
results = []
for seed in range(GAMES):
    _, jeff = rollout(seed, f"jeff:{adapter}", MAX_PIECES, 0.0, "notebook", choose=policy)
    _, teacher = rollout(seed, "teacher", MAX_PIECES, 0.0, "notebook")
    results.append((seed, jeff, teacher))
    print(f"seed {seed}: Jeff {jeff['pieces']} pieces / {jeff['lines']} lines ({jeff['terminal']}) | "
          f"teacher {teacher['pieces']} / {teacher['lines']}")

In [ ]:
ms = policy.server_ms
print(f"{len(ms)} moves, server mean {statistics.mean(ms):.1f} ms, p50 {statistics.median(ms):.1f} ms, "
      f"p95 {sorted(ms)[int(0.95 * (len(ms) - 1))]:.1f} ms")

In [ ]:
J.stop_server(proc)

## Next steps

Training for new tasks happens off the ANE (PyTorch on a Mac with MPS, or a GPU such as Colab). The resulting adapter and choice head are then merged, converted to Core AI, compiled and served on the Apple Neural Engine on the Mac (see the README, "Build your own"). Colab and other non-Apple machines cannot compile or serve on the ANE.

## Credits

- **Base model:** Jeff v1.3, a fine-tune of Qwen3.5-0.8B by its author, [mstrasser/jeff-base](https://huggingface.co/mstrasser/jeff-base) (Apache-2.0).
- **Backbone:** Qwen3.5-0.8B by the Qwen Team, Alibaba Cloud ([Qwen/Qwen3.5-0.8B](https://huggingface.co/Qwen/Qwen3.5-0.8B), Apache-2.0).
- **Snake adapter:** a rank-16 LoRA plus choice head trained for the base model on 2026-10-07 and merged before conversion; the build shipped here comes from a local Apple M5 Max run (PyTorch MPS, `forge.py jeff-train-lora --task snake`). A separate Unsloth Snake training on a Colab A100 (also 2026-10-07) exists as an experiment; it is not what this notebook serves.
- **Tetris adapter:** LoRA plus choice head trained locally on an Apple M5 Max with PyTorch MPS, using this repo's hand-rolled LoRA (no Unsloth), in the bounded overnight rounds of 2026-10-08 (round 02 shipped). A Colab A100 continuation on 2026-10-07 failed and was discarded.
- **Training data:** generated by this repo (El-Tetris oracle for Tetris, a search oracle for Snake); no third-party dataset.
- **Unsloth:** thanks to the [Unsloth](https://unsloth.ai) team. Jeff / Qwen3.5-0.8B LoRA adapters are compatible with Unsloth training (a Snake LoRA run with Unsloth on Colab A100 and T4 was verified), but the shipped adapters were not trained with Unsloth code or runtime. The training code here is Anemll's own, and the head training workflow was adopted from Unsloth's notebook.
- **Teacher heuristic:** El-Tetris by Islam El-Ashi (2011), an improvement on Pierre Dellacherie's algorithm — [imake.ninja/el-tetris-an-improvement-on-pierre-dellacheries-algorithm](https://imake.ninja/el-tetris-an-improvement-on-pierre-dellacheries-algorithm/).
- **Dependencies:** PyTorch, Transformers, safetensors, tokenizers, Hugging Face Hub, coremltools and the Apple Core AI packages (`coreai-core`, `coreai-opt`, `coreai-torch`) are listed in `requirements-inference.txt` (host venv: PyTorch, Transformers, safetensors, tokenizers, Hugging Face Hub) and `requirements-conversion.txt` (Core AI SDK venv: Core AI packages, coremltools, PyTorch, safetensors).